In [2]:
!fusermount -u /content/drive  # unmount if mounted
!rm -rf /content/drive         # remove any leftover files

from google.colab import drive
drive.mount('/content/drive')

fusermount: failed to unmount /content/drive: No such file or directory
Mounted at /content/drive


Split data into test and val (75-25) from the train data + yaml file to derive folder structure

In [ ]:
from pathlib import Path
import yaml

DST_ROOT = Path("/content/drive/MyDrive/Store/MOT20/MOT20-clean")

data_yaml = {
    "path": str(DST_ROOT),
    "train": str(DST_ROOT/"images/train"), # absolute path to train images
    "val":   str(DST_ROOT/"images/val"),
    "test":   str(DST_ROOT/"images/test"),# absolute path to val images
    "nc": 1,
    "names": {0: "person"}
}

with open(DST_ROOT/"data.yaml", "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print("Wrote data.yaml at", DST_ROOT/"data.yaml")
print("Contents:")
print(yaml.safe_dump(data_yaml, sort_keys=False))


Wrote data.yaml at /content/drive/MyDrive/Store/MOT20/MOT20-clean/data.yaml
Contents:
path: /content/drive/MyDrive/Store/MOT20/MOT20-clean
train: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train
val: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/val
test: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/test
nc: 1
names:
  0: person



In [3]:
!cat /content/drive/MyDrive/Store/MOT20/MOT20-clean/data.yaml


path: /content/drive/MyDrive/Store/MOT20/MOT20-clean
train: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train
val: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/val
test: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/test
nc: 1
names:
  0: person


YOLO v8 (installation + training + validating)

In [4]:
!pip -q install ultralytics
!pip show ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.3 MB/s eta 0:00:00
Name: ultralytics
Version: 8.3.235
Summary: Ultralytics YOLO 🚀 for SOTA object detection, multi-object tracking, instance segmentation, pose estimation and image classification.
Home-page: https://ultralytics.com
Author: 
Author-email: Glenn Jocher <glenn.jocher@ultralytics.com>, Jing Qiu <jing.qiu@ultralytics.com>
License: AGPL-3.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: matplotlib, numpy, opencv-python, pillow, polars, psutil, pyyaml, requests, scipy, torch, torchvision, ultralytics-thop
Required-by: 


Training

In [6]:
from ultralytics import YOLO
import os

# make sure local project folder exists (faster than writing directly to Drive)
os.makedirs("/content/yolo_train", exist_ok=True)

# load base model
model = YOLO("yolov8n.pt")

# train
model.train(
    data="/content/drive/MyDrive/Store/MOT20/MOT20-clean/data.yaml",
    epochs=20,
    imgsz=320,
    batch=16,
    workers=2,
    device=0,            # GPU device
    freeze=10,
    patience=10,
    lr0=0.002,
    project="/content/yolo_train",
    name="yolov8n_mot20_gpu_headft_smallimg",
    exist_ok=True,
    save=True,
    save_period=5,
    amp=True
)

# after training, copy results back to Drive
!cp -r /content/yolo_train /content/drive/MyDrive/Store/



SystemError: -package>() method: bad call flags

In [ ]:
from ultralytics import YOLO
import os

# make sure local project folder exists (faster than writing directly to Drive)
os.makedirs("/content/yolo_train", exist_ok=True)

# load base model
model = YOLO("yolov8n.pt")

# train
model.train(
    data="/content/drive/MyDrive/Store/MOT20/MOT20-clean/data.yaml",
    epochs=100,
    imgsz=320,
    batch=16,
    workers=2,
    device=0,            # GPU device
    freeze=10,
    patience=300,
    lr0=0.002,
    project="/content/yolo_train",
    name="yolov8n_mot20_gpu_headft_smallimg2",
    exist_ok=True,
    save=True,
    save_period=5,
    amp=True
)

# after training, copy results back to Drive
!cp -r /content/yolo_train /content/drive/MyDrive/Store/

Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Store/MOT20/MOT20-clean/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_mot20_gpu_headft_smallimg2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tr

Metrics

In [ ]:
from ultralytics import YOLO

# Load trained YOLOv8 model
model = YOLO("/content/drive/MyDrive/Store/yolo_train/yolov8n_mot20_gpu_headft_smallimg/weights/best.pt")

# Evaluate on validation set
metrics = model.val(
    data="/content/drive/MyDrive/Store/MOT20/MOT20-clean/data.yaml",
    imgsz=640,
    batch=16,
    conf=0.1,
    augment=True
)

# Print results
print("Validation Metrics:")
print(f"mAP@0.5     = {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95= {metrics.box.map:.3f}")
print(f"Precision   = {metrics.box.mp:.3f}")
print(f"Recall      = {metrics.box.mr:.3f}")


Ultralytics 8.3.184 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 132.4±81.0 MB/s, size: 410.5 KB)


val: Scanning /content/drive/MyDrive/Store/MOT20/MOT20-clean/labels/val/MOT20-05.cache... 3315 images, 0 backgrounds, 0 corrupt: 100%|██████████| 3315/3315 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):   1%|          | 2/208 [00:34<53:17, 15.52s/it]  

WARNING ⚠️ NMS time limit 2.800s exceeded


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 208/208 [01:40<00:00,  2.07it/s]


                   all       3315     751330      0.788      0.746      0.807      0.383
Speed: 0.2ms preprocess, 2.1ms inference, 0.0ms loss, 3.4ms postprocess per image
Results saved to runs/detect/val2
Validation Metrics:
mAP@0.5     = 0.807
mAP@0.5:0.95= 0.383
Precision   = 0.788
Recall      = 0.746


Inference

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import json

# --- Setup Paths ---
DRIVE_ROOT = Path("/content/drive/MyDrive/Store")
DATA_ROOT = DRIVE_ROOT / "MOT20/MOT20-clean"
MODEL_PATH = DRIVE_ROOT / "L1model/yolo_train/yolov8n_mot20_gpu_headft_smallimg/weights/best.pt"
OUTPUT_DIR = DRIVE_ROOT / "L1model/yolo_inf"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Load Model ---
model = YOLO(MODEL_PATH)

# --- Run Inference ---
# Find all individual sequence folders (e.g., MOT20-01, MOT20-05) directly,
# regardless of whether they are in 'train' or 'val' subfolders.
images_root = DATA_ROOT / "images"
all_sequence_paths = [path for path in images_root.glob("*/*") if path.is_dir()]

print("Starting inference on all found sequences...")

for seq_path in all_sequence_paths:
    seq_name = seq_path.name
    print(f"\nProcessing sequence: {seq_name}...")

    # Create a corresponding output folder for the results.
    seq_output_dir = OUTPUT_DIR / seq_name
    seq_output_dir.mkdir(parents=True, exist_ok=True)

    # Get all image files from the current sequence folder.
    image_files = sorted(seq_path.glob("*.jpg"))

    if not image_files:
        print(f"  -> WARNING: No images found in {seq_path}, skipping.")
        continue

    # Process each image in the sequence.
    for img_path in image_files:
        try:
            # Perform prediction on the image.
            result = model.predict(str(img_path), imgsz=640, conf=0.25, verbose=False)[0]

            detections = []
            if result.boxes is not None:
                for box in result.boxes:
                    # Filter for the 'person' class (ID 0).
                    if int(box.cls) == 0:
                        x1, y1, x2, y2 = box.xyxy[0].tolist()
                        confidence = float(box.conf)
                        detections.append([x1, y1, x2, y2, confidence])

            # Save the detection results to a JSON file if any were found.
            if detections:
                frame_name = img_path.stem
                output_json_path = seq_output_dir / f"{frame_name}.json"
                with open(output_json_path, "w") as f:
                    json.dump(detections, f)

        except Exception as e:
            # Catch errors on specific images without crashing the script.
            print(f"  -> ERROR processing file: {img_path}")
            print(f"     Details: {e}")
            continue # Skip to the next image

print(f"\nInference complete. Results saved in {OUTPUT_DIR}")


Starting inference on all found sequences...

Processing sequence: MOT20-05...

Processing sequence: MOT20-01...
  -> ERROR processing file: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000002.jpg
     Details: OpenCV(4.12.0) /io/opencv/modules/imgcodecs/src/loadsave.cpp:1266: error: (-215:Assertion failed) !buf.empty() in function 'imdecode_'

  -> ERROR processing file: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000004.jpg
     Details: OpenCV(4.12.0) /io/opencv/modules/imgcodecs/src/loadsave.cpp:1266: error: (-215:Assertion failed) !buf.empty() in function 'imdecode_'

  -> ERROR processing file: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000007.jpg
     Details: OpenCV(4.12.0) /io/opencv/modules/imgcodecs/src/loadsave.cpp:1266: error: (-215:Assertion failed) !buf.empty() in function 'imdecode_'

  -> ERROR processing file: /content/drive/MyDrive/Store/MOT20/MOT20-clean/images/train/MOT20-01/000009.jpg
     

Test

In [ ]:
from ultralytics import YOLO

# Load your trained model
model = YOLO("/content/drive/MyDrive/Store/L1model/yolo_train/yolov8n_mot20_gpu_headft_smallimg/weights/best.pt")

# Run evaluation on test set
metrics = model.val(
    data="/content/drive/MyDrive/Store/MOT20/MOT20-clean/data.yaml",
    split="test",
    imgsz=640,
    batch=16,
    conf=0.25
)

print("Test Set Metrics:")
print(f"mAP@0.5      = {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95 = {metrics.box.map:.3f}")
print(f"Precision    = {metrics.box.mp:.3f}")
print(f"Recall       = {metrics.box.mr:.3f}")


Ultralytics 8.3.225 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.7±0.3 ms, read: 39.2±16.1 MB/s, size: 70.9 KB)
val: Scanning /content/drive/MyDrive/Store/MOT20/MOT20-clean/labels/test/MOT20-04.cache... 3894 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3894/3894 5.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 244/244 2.4it/s 1:40
                   all       3894     727254      0.324      0.346      0.256     0.0733
Speed: 1.1ms preprocess, 4.1ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to /content/runs/detect/val3
Test Set Metrics:
mAP@0.5      = 0.256
mAP@0.5:0.95 = 0.073
Precision    = 0.324
Recall       = 0.346


In [ ]:
import cv2, json, matplotlib.pyplot as plt
from pathlib import Path

frame_id = "000120"
base = Path("/content/layer1_outputs")

det_paths = {
    "YOLOv8": base/"yolo_dets"/f"{frame_id}.json",
    "DETR":   base/"deform_detr_dets"/f"{frame_id}.json",
    "WBF":    base/"wbf_fused_dets"/f"{frame_id}.json"
}

img = cv2.imread(str(base/"frames_for_farneback"/f"{frame_id}.jpg"))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1,3,figsize=(15,5))
for ax,(title,p) in zip(axes,det_paths.items()):
    im = img_rgb.copy()
    if p.exists():
        for x1,y1,x2,y2,_,_ in json.load(open(p)):
            cv2.rectangle(im,(int(x1),int(y1)),(int(x2),int(y2)),(0,255,0),2)
    ax.imshow(im)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


error: OpenCV(4.12.0) /io/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'
